In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
seed = 14

df = pd.read_csv("./data_set.csv", sep="\t")



train, test = train_test_split(df, test_size=0.2, random_state=seed)

test.reset_index(inplace=True)
test.drop(columns=["index"], inplace=True)


In [2]:
df_qwen     = pd.read_csv("./testes/preds_qwen.txt", sep="\t", header=None).rename(columns={0: "qwen_outs"}, inplace=False)
df_final = pd.concat([df_qwen, test], axis=1)

In [3]:
from transformers import AutoTokenizer, AutoModel
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TextClassificationPipeline


def compute_acc(candidates):

    model     = AutoModelForSequenceClassification.from_pretrained("../dataset/data/ustance/classification/results/checkpoint-520/")
    tokenizer = AutoTokenizer.from_pretrained("neuralmind/bert-base-portuguese-cased")
    tokenizer.padding_side = "right"  # Fix weird overflow issue with fp16 training
    model = TextClassificationPipeline(model=model, tokenizer=tokenizer, top_k=1)
    preds = []
    for pred in model(candidates, truncation=True, padding=True, max_length=512):
        preds.append(int(pred[0]["label"].split("LABEL_")[1]))
    
    return preds

df_final["preds"] = compute_acc(df_final["qwen_outs"].tolist())

Device set to use cuda:0


In [4]:
labels = {  'r2_lu_876': 0,
            'r2_cl_2165': 1,
            'r2_gl_1071': 2,
            'r2_bo_344': 3,
            'r2_bo_208': 4,
            'r2_lu_94': 5}

In [5]:
"""convert columns user_id with the dicitonary labels"""
df_final["label"] = df_final["user_id"].apply(lambda x: labels[x] if x in labels else -1)

In [6]:
df_final["iscorrect"]  = df_final["preds"] == df_final["label"]

In [7]:
# Função para extrair um exemplo de acerto e um de erro por usuário
def extrair_acertos_erros_por_usuario(df):
    exemplos = []

    for user_id in df['user_id'].unique():
        df_user = df[df['user_id'] == user_id]

        # Um exemplo correto (label == 1)
        acertos = df_user[df_user['iscorrect'] == True]
        if not acertos.empty:
            exemplo_acerto = acertos.sample(1, random_state=seed)
            exemplos.append(exemplo_acerto)

        # Um exemplo incorreto (label == 0)
        erros = df_user[df_user['iscorrect'] == False]
        if not erros.empty:
            exemplo_erro = erros.sample(1, random_state=seed)
            exemplos.append(exemplo_erro)

    resultado = pd.concat(exemplos).reset_index(drop=True)
    return resultado

In [8]:
df_to_be_analised = extrair_acertos_erros_por_usuario(df_final)

In [9]:
df_to_be_analised

,qwen_outs,reescrito,estilo_academico,user_id,estilo_autoral,preds,label,iscorrect
0,Destaque na imprensa alemã,"<think>\nOkay, let's tackle this text rewritin...",Destaque marcante no jornalismo alemão,r2_bo_208,Que belo dia para o jornalismo alemão,5,4,False
1,Terrorismo: grupo ligado à Al Qaeda reivindica...,"<think>\nOkay, let's tackle this rewriting tas...",Um grupo jihadista associado à Al Qaeda reivin...,r2_lu_876,Grupo jihadista vinculado à Al Qaeda reivindi...,0,0,True
2,Que a luz abençoe a todos povo de DEUS,"<think>\nOkay, the user wants me to rewrite th...",Que haja abundância de luz para o povo de Deus.,r2_lu_876,Muita luz para o povo de D'us !!!,2,0,False
3,O dia 13 de Fevereiro de 2020 será lembrado na...,"<think>\nOkay, the user wants me to rewrite th...",O evento em 13 de fevereiro de 2020 pode ser c...,r2_cl_2165,ACABOU PRA VOCÊ BERGOGLIO!!!... ESTE DIA 13/0...,1,1,True
4,Ypiranga é posição certa - reforma administrat...,"<think>\nOkay, the user wants me to rephrase t...",O 'Posto Ypiranga' representa uma posição corr...,r2_cl_2165,O “POSTO YPIRANGA” ESTÁ CERTO!!!... HÁ QUE FA...,3,1,False
5,LINDA AINDA MAIS VERSUS DORES DO INDAIÁ/MG,"<think>\nOkay, let's tackle this rewriting tas...","Tristemente, por mais uma vez, a cidade de Dor...",r2_gl_1071,"Infelizmente, mais uma vez, Dores do Indaiá/M...",2,2,True
6,Esse Lula enganou milhões de eleitores por ano...,"<think>\nOkay, the user wants me to rewrite th...",Esse Lula enganou milhões de eleitores por ano...,r2_gl_1071,"Esse Lula enganou milhões de eleitores, por a...",3,2,False
7,Que um novo Pernambuco Revolução aconteça.,"<think>\nOkay, let's tackle this. The user wan...",Expressar o desejo de que outra Revolução Pern...,r2_bo_344,Rezando pra outra Revolução Pernambucana acon...,3,3,True
8,Quero a música Rajadao no Spotify pra eu posta...,"<think>\nOkay, let's tackle this rewriting tas...",Espera-se pela disponibilidade da música 'Raja...,r2_bo_344,"Esperando Rajadao sair no spotify, postar uma...",4,3,False
9,O PT não representa os trabalhadores mas sim a...,"<think>\nOkay, I need to rephrase the original...",O Partido dos Trabalhadores (PT) representa um...,r2_lu_94,"@ O PT é o reflexo de seu proprietário, Lula ...",3,5,False


In [10]:
from lime.lime_text import LimeTextExplainer
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
import torch

model     = AutoModelForSequenceClassification.from_pretrained("../dataset/data/ustance/classification/results/checkpoint-520/")
tokenizer = AutoTokenizer.from_pretrained("pablocosta/bertabaporu-base-uncased")

# Pipeline para predição
def predict_prob(texts):
    inputs = tokenizer(texts, padding=True, truncation=True, return_tensors="pt")
    outputs = model(**inputs)
    probs = torch.softmax(outputs.logits, dim=-1)
    return probs.detach().numpy()


In [ ]:
# Inicializa LIME
class_names = ["r2_lu_876", "r2_cl_2165", "r2_gl_1071", "r2_bo_344", "r2_bo_208", "r2_lu_94"]  # Substitua pelos nomes reais das classes
explainer = LimeTextExplainer(class_names=class_names)

# Textos simulados
textos_gerados = df_to_be_analised["qwen_outs"].tolist()

# Coleta explicações simuladas
tabela_resultado = []

for texto in textos_gerados:
    explicacao = explainer.explain_instance(
        text_instance=texto,
        classifier_fn=predict_prob,
        num_features=5
    )
    tokens_e_pesos = explicacao.as_list()

    tokens = [token for token, _ in tokens_e_pesos]
    pesos = [round(score, 4) for _, score in tokens_e_pesos]

    tabela_resultado.append({
        "Texto gerado": texto,
        "Tokens mais influentes": ", ".join(tokens),
        "Contribuição (LIME)": ", ".join(map(str, pesos))
    })

# Converte para DataFrame
df_resultado = pd.DataFrame(tabela_resultado)



Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


In [ ]:
df_resultado

,Texto gerado,Tokens mais influentes,Contribuição (LIME)
0,O Exército de Israel atira míssil Tammuz na fr...,"Síria, Israel, na, com, de","-0.0044, -0.004, 0.0018, 0.0016, 0.0016"
1,Que um novo Pernambuco Revolução aconteça.,"Pernambuco, um, novo, aconteça, Que","-0.0006, 0.0002, 0.0001, 0.0001, 0.0001"
